# AMD Radeon Fine-Tuning: Wayfarer-2-12B

This Unsloth configuration natively supports LLaMA-based architectures, accelerates training by 2x, and uses significantly less VRAM, which is perfect for an AMD GPU environment!

In [ ]:
!curl -O https://downloads.rclone.org/rclone-current-linux-amd64.zip
!unzip -q rclone-current-linux-amd64.zip
!sudo cp rclone-*-linux-amd64/rclone /usr/bin/
!sudo chown root:root /usr/bin/rclone
!sudo chmod 755 /usr/bin/rclone

import os
os.makedirs(os.path.expanduser('~/.config/rclone/'), exist_ok=True)
with open(os.path.expanduser('~/.config/rclone/rclone.conf'), 'w') as f:
    f.write('''[HetznerS3]
type = s3
provider = Other
access_key_id = YOUR_HETZNER_ACCESS_KEY
secret_access_key = YOUR_HETZNER_SECRET_KEY
endpoint = https://hel1.your-objectstorage.com
region = hel1
''')

# Download the judged datasets!
!mkdir -p /workspace/dataset
!rclone copy HetznerS3:pixeldata-cleaned/judged_and_cleaned/ /workspace/dataset/ -P

## Cell 2: Initialize Unsloth & Load Model
Unsloth automatically optimizes the model for 4-bit quantization, drastically reducing memory requirements.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 
dtype = None # Auto-detects float16 or bfloat16
load_in_4bit = True # Use 4bit quantization

# Wayfarer-2-12B is our base
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "LatitudeGames/Wayfarer-2-12B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Apply LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
)

## Cell 3: Prepare the ChatML Dataset
Load the JSONL files and map them into the ChatML format expected by the model.

In [ ]:
from datasets import load_dataset

# Load all downloaded jsonl files
dataset = load_dataset("json", data_files="/workspace/dataset/*.jsonl", split="train")

def format_chatml(example):
    texts = []
    for msg in example["messages"]:
        texts.append(f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n")
    return {"text": "".join(texts)}

dataset = dataset.map(format_chatml, num_proc=4)

## Cell 4: Train!
Start the SFT Trainer.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 4,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "wayfarer-outputs",
    ),
)

trainer.train()

## Cell 5: Export to GGUF
Export the finished model directly to a `.gguf` file or push it straight to HuggingFace.

In [ ]:
# Save LoRA locally
model.save_pretrained("wayfarer-2-12b-lora")
tokenizer.save_pretrained("wayfarer-2-12b-lora")

# Push directly to HuggingFace in 4-bit quantized GGUF format
model.push_to_hub_gguf(
    "your_hf_username/Wayfarer-2-12B-Pixelated",
    tokenizer,
    quantization_method = "q4_k_m",
    token = "YOUR_HF_TOKEN",
)